# 6.2 Companion: Seeing Optimisation in Practice

Reading about gradient descent is one thing. Watching a loss number fall -- epoch by epoch, in response to decisions you make -- is another.

This notebook runs a short training experiment on real data. The model is deliberately simple so it finishes in seconds, giving you a clear view of the process without the noise of a long run. Each step connects to a specific lesson from 6.2. Run the cells in order, and write down any predictions before you run rather than after.

**What this covers:**
- Why we split data into training and validation sets, and what each one tells us (Lesson 2)
- What gradient descent actually looks like epoch by epoch (Lesson 3)
- How learning rate choice shapes the curve (Lesson 4)
- Where convergence happens and how to read it (Lesson 5)


## Setup

Run this cell to import the libraries used throughout.


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# --- Training data: synthetic with label noise ---
candidates = [
    os.path.join("lab", "data", "synth_financial_sentiment.csv"),
    os.path.join("data", "synth_financial_sentiment.csv"),
    "synth_financial_sentiment.csv",
]
csv_path = next((p for p in candidates if os.path.exists(p)), None)
if csv_path is None:
    raise FileNotFoundError("Could not find synth_financial_sentiment.csv")

df = pd.read_csv(csv_path)
label2id = {"negative": 0, "neutral": 1, "positive": 2}
df["y"] = df["label"].map(label2id)
X_train = df["sentence"].to_numpy()
y_train = df["y"].to_numpy().copy()

# Inject label noise into training data only (25% random label flips)
rng = np.random.default_rng(42)
classes = np.array([0, 1, 2])
n_noisy = int(0.25 * len(y_train))
flip_idx = rng.choice(len(y_train), size=n_noisy, replace=False)
for idx in flip_idx:
    y_train[idx] = rng.choice([c for c in classes if c != y_train[idx]])

# --- Validation data: real financial_phrasebank (fallback to clean synthetic) ---
val_source = None
try:
    from datasets import load_dataset
    ds = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)["train"]
    val_df = ds.to_pandas().rename(columns={"sentence": "text", "label": "y"})
    # Sample 200 balanced examples
    val_df = val_df.groupby("y", group_keys=False).apply(lambda g: g.sample(min(67, len(g)), random_state=42))
    X_val = val_df["text"].to_numpy()
    y_val = val_df["y"].to_numpy()
    val_source = "financial_phrasebank (real financial news)"
except Exception:
    # Offline fallback: held-out portion of synthetic data (clean, no noise)
    _, X_val, _, y_val = train_test_split(
        df["sentence"].to_numpy(), df["y"].to_numpy(),
        test_size=0.2, random_state=42, stratify=df["y"].to_numpy()
    )
    val_source = "synthetic data held-out split (offline fallback)"

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec   = vectorizer.transform(X_val)

print(f"Training examples:   {len(X_train)} ({n_noisy} labels randomly corrupted)")
print(f"Validation examples: {len(X_val)} -- {val_source}")
print()
if "phrasebank" in val_source:
    print("Using real financial news as validation data.")
    print("Expect a meaningful gap between training and validation accuracy --")
    print("real sentences are harder than synthetic ones, and the training data")
    print("has noise the model may partially fit.")
else:
    print("Running offline: validation is a clean synthetic split.")
    print("The generalisation gap will be smaller than with real data.")


## Step 1 -- Load the data and create a train/validation split

You have likely already seen data split into a training set and a test set. The idea is straightforward: train on one portion, evaluate on a held-out portion the model has never seen, so you know whether it has learned something general rather than memorised the training data.

A validation set works the same way, but serves a different purpose. The test set is used once, at the very end, to report final performance. The validation set is used *during* training -- after every epoch -- to give you a running picture of how the model is performing on unseen data as it learns.

This matters because training loss alone is not a reliable signal. A model can improve on training data while getting worse on everything else. The validation loss tells you which is happening.

This notebook uses a deliberate combination:

- **Training set**: synthetic financial sentiment data (900 examples), with 25% of labels randomly corrupted. The noise is intentional -- it makes the model's task harder and produces a visible generalisation gap rather than the near-perfect scores you get from clean synthetic data.
- **Validation set**: real sentences from the `financial_phrasebank` dataset, sourced from actual financial news. If that download is unavailable, the notebook falls back to a clean held-out portion of the synthetic data.

Using a real validation set against a noisy training set is a reasonable approximation of what happens in practice: training data is often imperfect, and the validation set is your honest measure of whether anything useful was learned despite that.

Watch both curves in Step 3. If training and validation loss move together, the model is learning patterns that transfer. If they diverge, it is fitting the noise.


In [ ]:
import os

# Find the synthetic dataset
candidates = [
    os.path.join("lab", "data", "synth_financial_sentiment.csv"),
    os.path.join("data", "synth_financial_sentiment.csv"),
    "synth_financial_sentiment.csv",
]
csv_path = next((p for p in candidates if os.path.exists(p)), None)
if csv_path is None:
    raise FileNotFoundError("Could not find synth_financial_sentiment.csv. Run from the repo root.")

df = pd.read_csv(csv_path)
label2id = {"negative": 0, "neutral": 1, "positive": 2}
df["y"] = df["label"].map(label2id)

X_train, X_val, y_train, y_val = train_test_split(
    df["sentence"].to_numpy(), df["y"].to_numpy(),
    test_size=0.2, random_state=42, stratify=df["y"].to_numpy()
)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)

print(f"Training examples: {len(X_train)}")
print(f"Validation examples: {len(X_val)}")
print(f"Classes: negative=0, neutral=1, positive=2")


## Step 2 -- Train for 15 epochs and record the losses

Each epoch, the model makes predictions on the training data, calculates how wrong it was (loss), and adjusts its weights to do better next time. Then we measure loss on the validation set to see whether the improvement is real.

**Before you run:** predict what will happen to both curves.
- Will training loss and validation loss stay close together, or will one pull ahead?
- Will the improvement be roughly steady each epoch, or will it slow down over time?

Write your prediction down. The point of making it explicit is not to be right -- it is to notice where your mental model and the actual output differ.


In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import log_loss, accuracy_score

EPOCHS = 15
LEARNING_RATE = 0.01  # JUST_RIGHT -- try changing this in Step 5

clf = SGDClassifier(
    loss="log_loss",
    learning_rate="constant",
    eta0=LEARNING_RATE,
    alpha=1e-4,
    random_state=42,
)

train_losses, val_losses = [], []
train_accs, val_accs = [], []

clf.partial_fit(X_train_vec, y_train, classes=classes)

for epoch in range(EPOCHS):
    clf.partial_fit(X_train_vec, y_train)
    train_prob = clf.predict_proba(X_train_vec)
    val_prob   = clf.predict_proba(X_val_vec)
    train_losses.append(log_loss(y_train, train_prob))
    val_losses.append(log_loss(y_val, val_prob))
    train_accs.append(accuracy_score(y_train, clf.predict(X_train_vec)))
    val_accs.append(accuracy_score(y_val, clf.predict(X_val_vec)))

print(f"Training complete. Final validation accuracy: {val_accs[-1]:.3f}")


## Step 3 -- Read the curves

The plot shows training loss and validation loss across all 15 epochs.

Three things to look for:
1. **Direction** -- are both curves falling? That is the basic signal that learning is happening.
2. **Gap** -- is validation loss consistently above training loss? A small gap is normal. A widening gap means the model is fitting the training data rather than learning from it.
3. **Rate of change** -- is the improvement roughly constant, or is each epoch returning less than the last?


In [ ]:
from sklearn.metrics import accuracy_score

epochs_range = range(1, EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_range, train_losses, marker='o', label='Training loss')
ax1.plot(epochs_range, val_losses, marker='o', label='Validation loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss curves')
ax1.legend()

ax2.plot(epochs_range, train_accs, marker='o', label='Training accuracy')
ax2.plot(epochs_range, val_accs, marker='o', label='Validation accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy')
ax2.set_ylim(0, 1)
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Final training accuracy:   {train_accs[-1]:.3f}")
print(f"Final validation accuracy: {val_accs[-1]:.3f}")
print(f"Generalisation gap:        {train_accs[-1] - val_accs[-1]:+.3f}")
print()
print("The training set had noisy labels, so training accuracy reflects")
print("how well the model fits corrupted data. Validation accuracy is the")
print("real signal -- it reflects performance on clean, unseen examples.")


## Step 4 -- Find the convergence point

Look at the curve from Step 3. The loss is still falling at epoch 15, but the rate of improvement has changed across the run. Early epochs produce large drops. Later epochs produce smaller ones.

This is convergence in practice: not a sudden stop, but a gradual flattening. The question is not whether to stop training -- it is when the remaining improvement no longer justifies the compute cost.

A useful parallel: Many public EV chargers on public networks in a number of countries -- though not yet consistently in the UK -- cut off at 80%. The last 20% of charge takes roughly as long as the first 80% and generates more heat stress on the battery. The car is already useful. The same logic applies here -- at some point, another epoch is expensive and the gain is small.

The table below calculates the loss delta between consecutive epochs and flags where the curve is flattening.


In [ ]:
print(f"{'Epoch':>6} {'Train loss':>12} {'Delta':>10} {'Signal':>10}")
print("-" * 44)
for i in range(len(train_losses)):
    delta = abs(train_losses[i] - train_losses[i-1]) if i > 0 else float('inf')
    signal = "still learning" if delta > 0.01 else "flattening"
    print(f"{i+1:>6} {train_losses[i]:>12.4f} {delta:>10.4f} {signal:>12}")

print()
final_delta = abs(train_losses[-1] - train_losses[-2])
if final_delta < 0.01:
    print("The run appears to have converged. Further epochs unlikely to help much.")
else:
    print(f"Still improving at epoch {EPOCHS}. More epochs may help — but check the validation curve too.")


## Step 5 -- Change the learning rate and re-run

Go back to Step 2, change `LEARNING_RATE` to one of the values below, then re-run Steps 2, 3, and 4.

| Value | Label | What to expect |
|-------|-------|----------------|
| `0.0001` | TOO_LOW | Loss barely moves -- 15 epochs, almost no progress |
| `0.01` | JUST_RIGHT | Steady descent, training and validation loss track each other |
| `5.0` | TOO_HIGH | Loss oscillates or climbs rather than falling |

Try all three. After each run, note what the curve looks like and where (if anywhere) convergence occurs.

This is the same experiment you will run in Workshop 6W on a transformer model. The curve shapes are identical -- only the scale and training time differ. Going into that workshop having already seen these patterns means you will be diagnosing something familiar, not encountering it for the first time.

**One question to sit with before 6W:** which learning rate produced the smallest gap between training and validation loss at convergence, and why does that gap matter?


---

## Going deeper: common questions

These are not required reading. They are here for anyone who found the experiment raised more questions than it answered.

---

### Why do we need a validation set if we already have a training set and a test set?

The test set exists to answer one question at the very end: how does this model perform on data it has never seen? You use it once, report the number, and you are done. The problem is that if you make any decisions during training based on test set performance -- even small ones, like deciding when to stop -- you have effectively let the test set influence the model, and the final number is no longer a fair measure.

The validation set solves this by giving you a separate pool of unseen examples to consult as many times as you like during training, without contaminating the test set. Think of it as a rehearsal audience. The test set is the actual performance, used only once.

In short: training set teaches the model, validation set guides decisions during training, test set gives the honest final verdict.

---

### How does what I just saw connect to the 3D loss landscape diagrams in 6.2?

In the 3D diagrams, the landscape represents all possible combinations of model weights, and the height at any point represents the loss for those weights. Training is the process of moving across that landscape toward lower ground.

Each epoch in this notebook is one step across that landscape. The training loss you recorded is the height at each step -- how high up you still are after that many steps. When the training loss curve flattens, it means the steps are no longer moving you much lower. You are close to the bottom of the local valley you found.

The validation loss is the height at the same position, but measured on different ground -- the validation data. If the landscape looks similar from both perspectives, training and validation loss track together. If the model has found a valley that only exists in the training data (because of noise, or because it memorised specific examples), the training loss is low but the validation loss remains higher. That divergence is the gap you saw in your plot.

---

### How is the convergence cutoff decided in practice?

There is no single answer. In research, models are often trained until the validation loss stops improving for a fixed number of epochs -- this is called early stopping, and the patience parameter (how many non-improving epochs to tolerate before stopping) is set by the practitioner. A common default is 3 to 5 epochs of no improvement.

In production, the decision is usually made on business grounds. A model that achieves 91% accuracy after 10 epochs and 91.4% after 20 epochs probably does not justify the cost of the extra training, especially if the 0.4% difference is below the margin of error in evaluation. The cutoff is wherever the curve flattens enough that the remaining gain is not worth the compute cost, the time, or the environmental load.

What the delta table in Step 4 gives you is a simple version of this: a threshold below which the improvement is judged to be marginal. In real training pipelines, this threshold is a tunable parameter, not a fixed number.

---

### Is all engineering scientific? Or is some of it just trial and error?

Mostly the latter, more than practitioners admit. The mathematics of gradient descent is precise, but the decisions surrounding it -- which learning rate to start with, when to stop, which schedule to use, how much noise is acceptable in the training data -- are empirical. They are made by running experiments, observing results, and applying judgement built up over many previous experiments.

Senior ML engineers do not derive the optimal learning rate from first principles. They run hyperparameter sweeps, look at curves, and make calls based on experience. The science gives you the tools and the vocabulary. The engineering is knowing how to use them in the face of incomplete information and time pressure.

This is worth holding onto: being uncertain about a configuration choice and running an experiment to find out is not a sign of not knowing what you are doing. It is what knowing what you are doing looks like.

---

### What are underfitting and overfitting, and how do they relate to what I observed?

These are two failure modes that sit on opposite ends of the same spectrum. Understanding them is central to almost every model training decision.

**Underfitting** is when the model has not learned enough from the training data to make useful predictions on anything. The training loss is still high, the validation loss is still high, and performance on both is poor. The model has not found a useful part of the loss landscape yet.

A concrete example: imagine trying to predict whether a film review is positive or negative using only the length of the review in words. Length has almost no relationship to sentiment. No matter how long you train, the model cannot learn something that is not in the signal. The training loss will stay high because the model is fundamentally under-equipped for the task. This is underfitting.

In your curves, underfitting looks like both training loss and validation loss remaining stubbornly high across all epochs -- the TOO_LOW learning rate experiment often produces this, because the steps are so small that the model barely moves across the loss landscape.

**Overfitting** is the opposite problem: the model has learned the training data too well, including its noise and quirks, at the expense of learning anything general. Training loss is low, but validation loss is higher -- sometimes much higher. The model has memorised the training examples rather than extracting patterns that transfer to new data.

A concrete example: a model trained to identify spam emails on a very small dataset might learn that emails from a specific sender address are spam, because that address happened to appear in the training data. It performs perfectly on the training data, but when a different spammer uses a new address, the model fails. It memorised a coincidence rather than learning what spam looks like in general.

In your curves, overfitting looks like the gap between training and validation loss widening over time -- training loss keeps falling while validation loss flattens or rises. The model is fitting the noise in the training labels (which you deliberately injected) rather than the underlying patterns in the text.

The validation set is your main instrument for detecting overfitting early. Without it, you would only see the training loss falling and assume everything was going well. With it, you can see the moment the model starts to overfit and stop before it gets worse.

Most real training runs sit somewhere between these two extremes. The goal of choosing a good learning rate, the right amount of training data, an appropriate model size, and the right stopping point is to land in that middle ground: a model that has learned enough to generalise, but not so much that it has memorised the training set.
